# 16 · Monotonic Stack

A **monotonic stack** is a stack kept in sorted order: before pushing a new element we **pop** every element that would break that order. Each element is pushed once and popped at most once, so many "nearest bigger/smaller element" problems become **O(n)** instead of O(n²).

The code is exactly my original; I added explanations and comments.

**Topics:** next greater element (brute force → stack) · next greater element in a circular array · asteroid collision

## 1. Next Greater Element (NGE)
For every element find the **first larger element to its right**; if none exists, answer `-1`.
```
nums = [19, 4, 2, 3, 6, 5, 11, 3, 10, 12]
ans  = [-1, 6, 3, 6, 11, 11, 12, 10, 12, -1]
```
**Brute force:** for each `i` scan `j = i+1 …` until something bigger is found. O(n²).

In [ ]:
nums=[19,4,2,3,6,5,11,3,10,12]
# -1 is the default answer ("no greater element")
ans=[-1]*len(nums)
for i in range(0,len(nums)):
  # look at everything to the right of i
  for j in range(i+1,len(nums)):
    # the first greater element
    if nums[j]> nums[i]:
      ans[i]=nums[j]
      # stop at the first one
      break
print(ans)

[-1, 6, 3, 6, 11, 11, 12, 10, 12, -1]


## 2. NGE with a monotonic stack: O(n)
**Scan from right to left.** The stack holds *candidates* for "the next greater element", with values that get larger towards the bottom (the top is the closest/smallest candidate).

For each `nums[i]` (right → left):
1. **Pop** every stack element `<= nums[i]`; they are not bigger than `nums[i]` and `nums[i]` is closer to anything on the left, so they can never be the first greater element again.
2. If the stack is non-empty its **top** is the answer for `i`; otherwise `-1`.
3. **Push** `nums[i]` as a candidate for the elements to its left.

Start of the trace: `12` → empty stack, ans `-1`, push; `10` → top `12`, ans `12`, push; `3` → top `10`, ans `10`, push; `11` → pop `3`, pop `10`, top `12`, ans `12`, push.

**Time:** O(n) (each element is pushed/popped once) · **Space:** O(n)

In [ ]:
# next greater element
nums=[19,4,2,3,6,5,11,3,10,12]
ans=[-1]*len(nums)
# candidates for the next greater element
stack=[]
n= len(nums)
# scan right to left
for i in range(n-1,-1,-1):
  # pop everything not greater than nums[i]
  while len(stack)!=0 and stack[-1] <= nums[i]:
    stack.pop()
  if len(stack)!=0:
    # the closest remaining greater element
    ans[i]=stack[-1]
  # nums[i] may be the answer for elements further left
  stack.append(nums[i])
print(ans)


[-1, 6, 3, 6, 11, 11, 12, 10, 12, -1]


## 3. Next Greater Element II: circular array
The array is **circular**: after the last element the search continues from the first. Example `[1,2,1]` → `[2,-1,2]`.

**Trick:** walk over the array **twice** (indexes `2n−1 … 0`, using `i % n`). Answers are stored only for the real positions `i < n`; the second copy just fills the stack with wrapped-around candidates. **Time:** O(n).

In [ ]:
class Solution:
    def nextGreaterElements(self, nums: list[int]) -> list[int]:
        n = len(nums)
        result = [-1] * n    # Initialize results with -1
        stack = []           # Stack for potential NGEs

        # Loop twice to simulate circular array
        for i in range(2 * n - 1, -1, -1):
            # Maintain decreasing stack
            while stack and stack[-1] <= nums[i % n]:
                stack.pop()
            # Only set result during the first 'real' pass
            if i < n:
                if stack:
                    result[i] = stack[-1]
            # Push current value for next iteration checks
            stack.append(nums[i % n])
        return result

## 4. Asteroid collision
Each number is an asteroid: the **sign** is the direction (`+` right, `−` left), the **absolute value** is the size. When two meet, the smaller explodes (equal → both explode). Return the survivors.

Only a **right-moving** asteroid followed by a **left-moving** one can collide, and the one that matters is the most recent right-mover, which is what the stack stores. For each asteroid:
- **`> 0`** (moving right): push
- **`< 0`** (moving left): it fights the stack top while that top is a right-mover:
  - top **smaller** → top explodes (pop) and keep fighting
  - top **equal** → both explode (pop, nothing pushed)
  - top **bigger** → the new asteroid explodes (nothing pushed)
  - stack empty or top also moving left → it **survives** (push)

**Time:** O(n) · **Space:** O(n)

In [ ]:
#asteroids collision
nums=[4,7,1,1,1,2,-3,-1,-7,17,15,-18,-19]
n= len(nums)
stack=[]
for i in range(0,n):
  # moving right: nothing to its left can hit it
  if nums[i] > 0:
    stack.append(nums[i])
  else:
    # destroy smaller right-movers
    while len(stack) !=0 and stack[-1]>0 and stack[-1]<abs(nums[i]):
      stack.pop()
    # same size: both explode
    if len(stack)!=0 and stack[-1] == abs(nums[i]):
      stack.pop()
    # nothing left to hit → the left-mover survives
    elif len(stack)==0 or stack[-1]<0:
      stack.append(nums[i])
print(stack)

[-18, -19]


---
### ✅ Key takeaways
- **Monotonic stack pattern:** before pushing `x`, pop everything `x` makes useless → O(n).
- **Next greater to the right** → scan right → left.
- **Circular array** → loop `2n` times and use `i % n`.
- **Asteroids:** the stack holds the survivors so far; a new left-mover pops the ones it destroys.